# Data Cleaning — Audit et nettoyage des audiogrammes

Ce notebook documente chaque critère de nettoyage appliqué avant l'entraînement.  
Chaque cellule peut être exécutée indépendamment pour inspecter un critère spécifique.

**Critères hard reject** (record écarté) :
1. Données vides
2. Moins de 7 fréquences par oreille
3. Fréquences obligatoires manquantes (1k, 2k, 4k) ou absence de haute fréquence (6k/8k)
4. Valeurs hors range physiologique (< -10 ou > 120 dB HL)
5. Audiogramme de calibration (plat à 0 dB)
6. Coverage mismatch OG/OD (> 2 fréquences différentes entre les deux oreilles)
7. Pente inversée extrême (corrélation fréq/seuil < -0.85 et écart > 80 dB) — validé clinicien
8. Double encoche (forme en W : 2+ pics locaux > 20 dB) — validé clinicien

**Critères soft flag** (record conservé mais annoté) :
9. Point aberrant isolé (> 35 dB de l'interpolation de ses voisins)
10. Asymétrie inter-auriculaire extrême (> 60 dB à une même fréquence)

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

from src.evaluate import plot_audiogram

plt.rcParams['figure.dpi'] = 110
sns.set_style('whitegrid')

# ─── Changer ici selon ta source ─────────────────────────────────────────────
DATA_PATH = Path('../data')                          # dossier multi-fichiers
# DATA_PATH = Path('../CORTI_sample_audiograms_500.json')  # fichier unique
# ─────────────────────────────────────────────────────────────────────────────

# ── Constantes ────────────────────────────────────────────────────────────────
MIN_FREQS_PER_EAR           = 7
DB_MIN, DB_MAX              = -10.0, 120.0
MAX_INTERAURAL_ASYMMETRY_DB = 60.0
MAX_COVERAGE_MISMATCH_FREQS = 2
MANDATORY_FREQS             = {1000.0, 2000.0, 4000.0}
MANDATORY_HIGH_FREQS        = {6000.0, 8000.0}
CALIBRATION_MAX_DB          = 10.0
CALIBRATION_MAX_STD         = 2.0
ABERRANT_NEIGHBOR_DIFF_DB   = 35.0
REVERSE_SLOPE_MIN_RANGE_DB  = 80.0
REVERSE_SLOPE_MAX_CORR      = -0.85
DOUBLE_NOTCH_MIN_DEPTH_DB   = 20.0

# ── Fonctions de détection ────────────────────────────────────────────────────
def _find_aberrant_points(dots: dict) -> list:
    """Fréquences où le seuil s'écarte > 35 dB de l'interpolation log-linéaire des voisins."""
    if len(dots) < 3:
        return []
    freqs = sorted(dots.keys())
    log_freqs = np.log(freqs)
    dbs = [dots[f] for f in freqs]
    aberrant = []
    for i in range(1, len(freqs) - 1):
        t = (log_freqs[i] - log_freqs[i-1]) / (log_freqs[i+1] - log_freqs[i-1])
        interp = dbs[i-1] + t * (dbs[i+1] - dbs[i-1])
        if abs(dbs[i] - interp) > ABERRANT_NEIGHBOR_DIFF_DB:
            aberrant.append(freqs[i])
    return aberrant

def _has_extreme_reverse_slope(dots: dict) -> bool:
    """Pente inversée extrême : forte corrélation négative fréq/seuil ET écart > 80 dB."""
    if len(dots) < 4:
        return False
    freqs = sorted(dots.keys())
    dbs = [dots[f] for f in freqs]
    corr = float(np.corrcoef(np.log(freqs), dbs)[0, 1])
    return corr < REVERSE_SLOPE_MAX_CORR and (max(dbs) - min(dbs)) > REVERSE_SLOPE_MIN_RANGE_DB

def _has_double_notch(dots: dict) -> bool:
    """Double encoche (W) : 2+ pics locaux dépassant les voisins de > 20 dB."""
    if len(dots) < 5:
        return False
    freqs = sorted(dots.keys())
    dbs = [dots[f] for f in freqs]
    notches = [
        freqs[i] for i in range(1, len(freqs) - 1)
        if dbs[i] > dbs[i-1] + DOUBLE_NOTCH_MIN_DEPTH_DB
        and dbs[i] > dbs[i+1] + DOUBLE_NOTCH_MIN_DEPTH_DB
    ]
    return len(notches) >= 2

def audit_record(row) -> list:
    """Retourne la liste des problèmes détectés sur un record (HARD: rejet, SOFT: flag)."""
    issues = []
    dots_L, dots_R = row['dots_left'], row['dots_right']
    all_vals = list(dots_L.values()) + list(dots_R.values())

    if not all_vals:
        return ["HARD:données_vides"]

    if len(dots_L) < MIN_FREQS_PER_EAR or len(dots_R) < MIN_FREQS_PER_EAR:
        issues.append(f"HARD:freqs_insuffisantes (L={len(dots_L)}, R={len(dots_R)})")

    for dots, label in [(dots_L, 'OG'), (dots_R, 'OD')]:
        freqs = set(dots.keys())
        missing = MANDATORY_FREQS - freqs
        if missing:
            issues.append(f"HARD:freq_obligatoire_manquante_{label} {sorted(int(f) for f in missing)}")
        if not (freqs & MANDATORY_HIGH_FREQS):
            issues.append(f"HARD:haute_freq_manquante_{label}")

    oor = [v for v in all_vals if v < DB_MIN or v > DB_MAX]
    if oor:
        issues.append(f"HARD:hors_range {oor}")

    if max(all_vals) <= CALIBRATION_MAX_DB and float(np.std(all_vals)) < CALIBRATION_MAX_STD:
        issues.append("HARD:calibration_sweep")

    mismatch = len(set(dots_L.keys()).symmetric_difference(set(dots_R.keys())))
    if mismatch > MAX_COVERAGE_MISMATCH_FREQS:
        issues.append(f"HARD:coverage_mismatch ({mismatch} fréquences)")

    for dots, label in [(dots_L, 'OG'), (dots_R, 'OD')]:
        if _has_extreme_reverse_slope(dots):
            issues.append(f"HARD:pente_inversee_extreme_{label}")
        if _has_double_notch(dots):
            issues.append(f"HARD:double_encoche_{label}")

    # Soft flags
    ab_L = _find_aberrant_points(dots_L)
    ab_R = _find_aberrant_points(dots_R)
    if ab_L or ab_R:
        issues.append(f"SOFT:point_aberrant OG:{[int(f) for f in ab_L]} OD:{[int(f) for f in ab_R]}")

    common = set(dots_L.keys()) & set(dots_R.keys())
    if common:
        max_asym = max(abs(dots_L[f] - dots_R[f]) for f in common)
        if max_asym > MAX_INTERAURAL_ASYMMETRY_DB:
            issues.append(f"SOFT:asymetrie_interauriculaire ({max_asym:.0f} dB)")

    return issues

def _is_hard_reject(issues: list) -> bool:
    return any(i.startswith("HARD:") for i in issues)

def clean_dataset(df):
    all_issues = df.apply(audit_record, axis=1)
    hard_mask = all_issues.apply(_is_hard_reject)
    rejected_df = df[hard_mask].copy()
    rejected_df['rejection_reasons'] = all_issues[hard_mask].apply(
        lambda iss: ' | '.join(i for i in iss if i.startswith("HARD:"))
    )
    clean_df = df[~hard_mask].copy()
    clean_df['aberrant_flags'] = all_issues[~hard_mask].apply(
        lambda iss: ' | '.join(i for i in iss if i.startswith("SOFT:"))
    )
    return clean_df, rejected_df

def cleaning_report(df_raw, clean_df, rejected_df):
    n = len(df_raw)
    print(f"Records bruts      : {n}")
    print(f"Records conservés  : {len(clean_df)} ({100*len(clean_df)/n:.1f}%)")
    print(f"Records rejetés    : {len(rejected_df)} ({100*len(rejected_df)/n:.1f}%)")
    soft = (clean_df.get('aberrant_flags', pd.Series('', index=clean_df.index)) != '').sum()
    print(f"Soft flags         : {soft} ({100*soft/max(len(clean_df), 1):.1f}% des conservés)")

## 0. Chargement brut (sans filtre)

In [ ]:
# Bypass du filtre test intégré au loader pour tout voir
import src.loader as _loader
_orig = _loader._is_test_audiogram
_loader._is_test_audiogram = lambda L, R: False

if DATA_PATH.is_dir():
    from src.loader import load_dataset
    df_raw = load_dataset(DATA_PATH)
else:
    from src.loader import load_json_file
    df_raw = pd.DataFrame(load_json_file(DATA_PATH)).sort_values(['patient','visit_date']).reset_index(drop=True)

_loader._is_test_audiogram = _orig

print(f"Records chargés    : {len(df_raw)}")
print(f"Patients uniques   : {df_raw['patient'].nunique()}")
if 'source_file' in df_raw.columns:
    print(f"Fichiers sources   : {df_raw['source_file'].nunique()}")
    print(df_raw.groupby('source_file').size().rename('n_records').to_string())

## 1. Distribution du nombre de fréquences par oreille

Le minimum clinique retenu est **7 fréquences** par oreille  
(protocole québécois : 500, 1000, 2000, 3000, 4000, 6000, 8000 Hz)

In [ ]:
n_freqs_L = df_raw['dots_left'].apply(len)
n_freqs_R = df_raw['dots_right'].apply(len)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for ax, vals, label in zip(axes, [n_freqs_L, n_freqs_R], ['OG (gauche)', 'OD (droite)']):
    counts = vals.value_counts().sort_index()
    colors = ['tomato' if n < MIN_FREQS_PER_EAR else 'steelblue' for n in counts.index]
    bars = ax.bar(counts.index.astype(str), counts.values, color=colors, edgecolor='white', alpha=0.9)
    ax.axvline(x=counts.index.tolist().index(MIN_FREQS_PER_EAR) - 0.5
               if MIN_FREQS_PER_EAR in counts.index else -1,
               color='red', linestyle='--', linewidth=1.5, label=f'Minimum : {MIN_FREQS_PER_EAR}')
    for bar, val in zip(bars, counts.values):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
                str(val), ha='center', fontsize=8)
    ax.set_xlabel('Nombre de fréquences mesurées')
    ax.set_ylabel('Nombre de records')
    ax.set_title(f'Couverture fréquentielle — {label}')
    ax.legend()

rejected_L = (n_freqs_L < MIN_FREQS_PER_EAR).sum()
rejected_R = (n_freqs_R < MIN_FREQS_PER_EAR).sum()
print(f"Records < {MIN_FREQS_PER_EAR} fréquences OG : {rejected_L} ({100*rejected_L/len(df_raw):.1f}%)")
print(f"Records < {MIN_FREQS_PER_EAR} fréquences OD : {rejected_R} ({100*rejected_R/len(df_raw):.1f}%)")
plt.tight_layout()
plt.show()

## 2. Fréquences présentes dans le dataset

Vérification que les fréquences obligatoires (1k, 2k, 4k + 6k/8k) sont bien couvertes.

In [ ]:
from collections import Counter

freq_counter_L = Counter()
freq_counter_R = Counter()
for _, row in df_raw.iterrows():
    freq_counter_L.update(row['dots_left'].keys())
    freq_counter_R.update(row['dots_right'].keys())

all_freqs = sorted(set(freq_counter_L) | set(freq_counter_R))
n = len(df_raw)

fig, ax = plt.subplots(figsize=(13, 4))
x = range(len(all_freqs))
pct_L = [100 * freq_counter_L.get(f, 0) / n for f in all_freqs]
pct_R = [100 * freq_counter_R.get(f, 0) / n for f in all_freqs]
ax.bar([xi - 0.2 for xi in x], pct_L, width=0.4, label='OG', color='steelblue', alpha=0.85)
ax.bar([xi + 0.2 for xi in x], pct_R, width=0.4, label='OD', color='coral', alpha=0.85)
ax.axhline(95, color='red', linestyle='--', linewidth=1, label='95% couverture')
ax.set_xticks(list(x))
ax.set_xticklabels([f'{int(f)}' if f < 1000 else f'{int(f)//1000}k' for f in all_freqs])
ax.set_ylabel('% records avec cette fréquence')
ax.set_title('Couverture par fréquence (OG et OD)')
ax.legend()
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

print('\nFréquences obligatoires :')
for f in sorted(MANDATORY_FREQS | MANDATORY_HIGH_FREQS):
    pL = 100 * freq_counter_L.get(f, 0) / n
    pR = 100 * freq_counter_R.get(f, 0) / n
    status = '✓' if pL > 90 and pR > 90 else '⚠'
    print(f'  {status} {int(f):5d} Hz — OG: {pL:.1f}%  OD: {pR:.1f}%')

## 3. Calibration sweeps (audiogrammes à 0 dB)

Audiogrammes où tous les seuils ≤ 10 dB avec std < 2 dB → sweep de calibration de l'audiomètre.

In [ ]:
def is_calibration(row):
    vals = list(row['dots_left'].values()) + list(row['dots_right'].values())
    return bool(vals) and max(vals) <= CALIBRATION_MAX_DB and np.std(vals) < CALIBRATION_MAX_STD

calib_mask = df_raw.apply(is_calibration, axis=1)
print(f"Calibration sweeps : {calib_mask.sum()} ({100*calib_mask.mean():.1f}%)")

if calib_mask.sum() > 0:
    n_show = min(calib_mask.sum(), 3)
    fig, axes = plt.subplots(1, n_show, figsize=(6*n_show, 4))
    if n_show == 1: axes = [axes]
    for ax, (_, row) in zip(axes, df_raw[calib_mask].head(n_show).iterrows()):
        vals = list(row['dots_left'].values()) + list(row['dots_right'].values())
        plot_audiogram(row['dots_left'], row['dots_right'],
                       title=f"Calibration — std={np.std(vals):.1f} dB", ax=ax)
    plt.suptitle('Exemples de calibration sweeps (rejetés)', color='red')
    plt.tight_layout()
    plt.show()

## 4. Coverage mismatch OG / OD

Rejeté si les deux oreilles n'ont pas le même ensemble de fréquences à ±2 fréquences près.  
Un bilan propre mesure toujours les deux oreilles aux mêmes fréquences.

In [ ]:
def coverage_mismatch(row):
    L = set(row['dots_left'].keys())
    R = set(row['dots_right'].keys())
    return len(L.symmetric_difference(R))

mismatch_counts = df_raw.apply(coverage_mismatch, axis=1)
reject_mismatch = mismatch_counts > MAX_COVERAGE_MISMATCH_FREQS

fig, ax = plt.subplots(figsize=(9, 4))
counts = mismatch_counts.value_counts().sort_index()
colors = ['tomato' if n > MAX_COVERAGE_MISMATCH_FREQS else 'steelblue' for n in counts.index]
ax.bar(counts.index.astype(str), counts.values, color=colors, edgecolor='white', alpha=0.9)
ax.axvline(x=counts.index.tolist().index(MAX_COVERAGE_MISMATCH_FREQS) + 0.5
           if MAX_COVERAGE_MISMATCH_FREQS in counts.index else -1,
           color='red', linestyle='--', linewidth=1.5, label=f'Seuil rejet > {MAX_COVERAGE_MISMATCH_FREQS}')
ax.set_xlabel('Nb fréquences différentes entre OG et OD')
ax.set_ylabel('Nombre de records')
ax.set_title('Coverage mismatch OG / OD')
ax.legend()
plt.tight_layout()
plt.show()

print(f"Coverage mismatch > {MAX_COVERAGE_MISMATCH_FREQS} : {reject_mismatch.sum()} records ({100*reject_mismatch.mean():.1f}%)")

if reject_mismatch.sum() > 0:
    for _, row in df_raw[reject_mismatch].head(5).iterrows():
        L = set(row['dots_left'].keys())
        R = set(row['dots_right'].keys())
        diff = L.symmetric_difference(R)
        print(f"  patient={str(row['patient'])[:16]} | OG seulement: {sorted(L-R)} | OD seulement: {sorted(R-L)}")

## 5. Valeurs hors range physiologique

Seuils hors [-10, 120] dB HL → impossibles à l'audiomètre standard.

In [ ]:
def out_of_range(row):
    vals = list(row['dots_left'].values()) + list(row['dots_right'].values())
    return [v for v in vals if v < DB_MIN or v > DB_MAX]

oor = df_raw.apply(out_of_range, axis=1)
oor_mask = oor.apply(bool)
print(f"Records hors range [{DB_MIN}, {DB_MAX}] dB : {oor_mask.sum()} ({100*oor_mask.mean():.1f}%)")

if oor_mask.sum() > 0:
    for _, row in df_raw[oor_mask].iterrows():
        print(f"  patient={str(row['patient'])[:16]} | valeurs aberrantes: {out_of_range(row)}")
else:
    print("  Aucun record hors range — données propres sur ce critère.")

## 6. Asymétrie inter-auriculaire extrême

Soft flag si |OG_f - OD_f| > 60 dB à une même fréquence.  
Peut être réel (surdité unilatérale) mais mérite vérification.

In [ ]:
def max_interaural_asym(row):
    common = set(row['dots_left'].keys()) & set(row['dots_right'].keys())
    if not common:
        return 0.0
    return max(abs(row['dots_left'][f] - row['dots_right'][f]) for f in common)

asym = df_raw.apply(max_interaural_asym, axis=1)
asym_flag = asym > MAX_INTERAURAL_ASYMMETRY_DB

fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(asym, bins=40, color='steelblue', edgecolor='white', alpha=0.85)
ax.axvline(MAX_INTERAURAL_ASYMMETRY_DB, color='orange', linestyle='--', linewidth=2,
           label=f'Seuil soft flag : {MAX_INTERAURAL_ASYMMETRY_DB} dB')
ax.set_xlabel('Asymétrie inter-auriculaire max (dB)')
ax.set_ylabel('Nombre de records')
ax.set_title('Distribution asymétrie OG/OD maximale')
ax.legend()
plt.tight_layout()
plt.show()

print(f"Asymétrie > {MAX_INTERAURAL_ASYMMETRY_DB} dB : {asym_flag.sum()} records ({100*asym_flag.mean():.1f}%) → soft flag")

if asym_flag.sum() > 0:
    n_show = min(asym_flag.sum(), 4)
    fig, axes = plt.subplots(1, n_show, figsize=(6*n_show, 4))
    if n_show == 1: axes = [axes]
    for ax, (_, row) in zip(axes, df_raw[asym_flag].head(n_show).iterrows()):
        plot_audiogram(row['dots_left'], row['dots_right'],
                       title=f"Asymétrie max={max_interaural_asym(row):.0f} dB", ax=ax)
    plt.suptitle(f'Records avec asymétrie > {MAX_INTERAURAL_ASYMMETRY_DB} dB (soft flag)', color='darkorange')
    plt.tight_layout()
    plt.show()

## 7. Points aberrants isolés

Soft flag si un seuil s'écarte de > 35 dB de l'interpolation log-linéaire de ses voisins.  
Indique un artefact de mesure, pas une pathologie.

In [ ]:
aberrant_mask = df_raw.apply(
    lambda row: bool(_find_aberrant_points(row['dots_left'])) or
                bool(_find_aberrant_points(row['dots_right'])), axis=1
)
print(f"Records avec points aberrants : {aberrant_mask.sum()} ({100*aberrant_mask.mean():.1f}%) → soft flag")

if aberrant_mask.sum() > 0:
    n_show = min(aberrant_mask.sum(), 4)
    fig, axes = plt.subplots(1, n_show, figsize=(6*n_show, 4))
    if n_show == 1: axes = [axes]
    for ax, (_, row) in zip(axes, df_raw[aberrant_mask].head(n_show).iterrows()):
        ab_L = _find_aberrant_points(row['dots_left'])
        ab_R = _find_aberrant_points(row['dots_right'])
        plot_audiogram(row['dots_left'], row['dots_right'],
                       title=f"Aberrant L:{[int(f) for f in ab_L]} R:{[int(f) for f in ab_R]}", ax=ax)
    plt.suptitle('Records avec points aberrants (soft flag — conservés)', color='darkorange')
    plt.tight_layout()
    plt.show()

## 8. Pente inversée extrême

Hard reject si la corrélation log(fréquence)/seuil est < -0.85 **et** l'écart min-max dépasse 80 dB.  
Validé cliniquement : un profil 120 dB → 20 dB linéaire sur toute la plage est un artefact (masking, erreur de saisie).

In [ ]:
reverse_L = df_raw['dots_left'].apply(_has_extreme_reverse_slope)
reverse_R = df_raw['dots_right'].apply(_has_extreme_reverse_slope)
reverse_mask = reverse_L | reverse_R

print(f"Pente inversée extrême : {reverse_mask.sum()} records ({100*reverse_mask.mean():.1f}%) → hard reject")

if reverse_mask.sum() > 0:
    n_show = min(reverse_mask.sum(), 4)
    fig, axes = plt.subplots(1, n_show, figsize=(6*n_show, 4))
    if n_show == 1: axes = [axes]
    for ax, (_, row) in zip(axes, df_raw[reverse_mask].head(n_show).iterrows()):
        plot_audiogram(row['dots_left'], row['dots_right'], title="Pente inversée extrême", ax=ax)
    plt.suptitle('Records pente inversée extrême (hard reject)', color='red')
    plt.tight_layout()
    plt.show()
else:
    print("  Aucun record avec pente inversée extrême.")

## 9. Double encoche — forme en W

Hard reject si 2+ pics locaux (perte > 20 dB par rapport aux deux voisins) sur la même oreille.  
Validé cliniquement : une double encoche n'existe pas en pathologie réelle — artefact de mesure.

In [ ]:
double_notch_L = df_raw['dots_left'].apply(_has_double_notch)
double_notch_R = df_raw['dots_right'].apply(_has_double_notch)
double_notch_mask = double_notch_L | double_notch_R

print(f"Double encoche (W) : {double_notch_mask.sum()} records ({100*double_notch_mask.mean():.1f}%) → hard reject")

if double_notch_mask.sum() > 0:
    n_show = min(double_notch_mask.sum(), 4)
    fig, axes = plt.subplots(1, n_show, figsize=(6*n_show, 4))
    if n_show == 1: axes = [axes]
    for ax, (_, row) in zip(axes, df_raw[double_notch_mask].head(n_show).iterrows()):
        plot_audiogram(row['dots_left'], row['dots_right'], title="Double encoche (W)", ax=ax)
    plt.suptitle('Records double encoche — forme en W (hard reject)', color='red')
    plt.tight_layout()
    plt.show()
else:
    print("  Aucun record avec double encoche.")

## 10. Nettoyage complet — rapport final

In [ ]:
clean_df, rejected_df = clean_dataset(df_raw)
cleaning_report(df_raw, clean_df, rejected_df)

In [ ]:
# Visualisation synthétique
n_clean    = len(clean_df)
n_rejected = len(rejected_df)
n_soft     = (clean_df.get('aberrant_flags', pd.Series('', index=clean_df.index)) != '').sum()
n_total    = len(df_raw)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Camembert global
sizes  = [n_clean - n_soft, n_soft, n_rejected]
labels = [f'Propres ({n_clean - n_soft})', f'Soft flag ({n_soft})', f'Rejetés ({n_rejected})']
colors = ['#2ecc71', '#f39c12', '#e74c3c']
axes[0].pie(sizes, labels=labels, colors=colors, autopct='%1.1f%%', startangle=90)
axes[0].set_title(f'Résultat du nettoyage (n={n_total})')

# Raisons de rejet
if n_rejected > 0:
    all_reasons = rejected_df['rejection_reasons'].str.split(' | ', regex=False).explode()
    reason_prefix = all_reasons.str.split(':').str[0].value_counts()
    reason_prefix.plot(kind='barh', ax=axes[1], color='tomato', edgecolor='white', alpha=0.85)
    axes[1].set_xlabel('Nombre de records')
    axes[1].set_title('Raisons de rejet')
    axes[1].grid(axis='x', alpha=0.3)
else:
    axes[1].text(0.5, 0.5, 'Aucun rejet', ha='center', va='center', fontsize=14)
    axes[1].axis('off')

plt.tight_layout()
plt.show()

In [ ]:
if 'source_file' in df_raw.columns:
    all_issues = df_raw.apply(audit_record, axis=1)
    hard_mask = all_issues.apply(_is_hard_reject)
    soft_mask = (~hard_mask) & all_issues.apply(bool)

    df_raw['status'] = 'propre'
    df_raw.loc[hard_mask, 'status'] = 'rejeté'
    df_raw.loc[soft_mask, 'status'] = 'soft flag'

    pivot = df_raw.groupby(['source_file', 'status']).size().unstack(fill_value=0)
    pivot_pct = pivot.div(pivot.sum(axis=1), axis=0) * 100

    ax = pivot_pct.plot(kind='bar', figsize=(12, 4), color=['#2ecc71', '#e74c3c', '#f39c12'],
                        edgecolor='white', alpha=0.85)
    ax.set_xlabel('')
    ax.set_ylabel('% records')
    ax.set_title('Qualité des données par fichier source')
    ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right')
    ax.legend(title='Statut')
    ax.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    plt.show()

    print(pivot.to_string())

## 11. Visualisation des records rejetés

Inspecter les audiogrammes rejetés pour valider que les critères sont bien calibrés.

In [ ]:
if len(rejected_df) > 0:
    n_show = min(len(rejected_df), 6)
    cols = 3
    rows = (n_show + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(14, 5*rows))
    axes = axes.flatten() if rows > 1 else [axes] if cols == 1 else axes.flatten()

    for i, (_, row) in enumerate(rejected_df.head(n_show).iterrows()):
        reason = row['rejection_reasons'].split(' | ')[0][:40]
        plot_audiogram(row['dots_left'], row['dots_right'],
                       title=f"REJETÉ\n{reason}", ax=axes[i])
        axes[i].set_facecolor('#fff5f5')

    for j in range(i+1, len(axes)):
        axes[j].axis('off')

    plt.suptitle(f'Aperçu des records rejetés (sur {len(rejected_df)} total)', fontsize=12, color='red')
    plt.tight_layout()
    plt.show()
else:
    print('Aucun record rejeté.')

## 12. Export du rapport de nettoyage

In [ ]:

# Export CSV des records rejetés pour archivage
if len(rejected_df) > 0:
    out_path = Path('../data/rejected_records.csv')
    export_cols = [c for c in ['source_file', 'patient', 'visit_date', 'visit_category',
                               'n_freqs_left', 'n_freqs_right', 'rejection_reasons']
                   if c in rejected_df.columns]
    rejected_df[export_cols].to_csv(out_path, index=False)
    print(f"Rejetés exportés → {out_path}")

# Sauvegarde du dataset propre en JSON
clean_path = Path('../data/clean_dataset.json')
clean_df.to_json(clean_path, orient='records', date_format='iso', force_ascii=False, indent=2)
print(f"Dataset propre sauvegardé → {clean_path}")
print(f"  Rechargement : python main.py --data data/clean_dataset.json")

# Résumé final
print(f"\n{'='*40}")
print(f"Dataset propre prêt pour l'entraînement")
print(f"{'='*40}")
print(f"  Records conservés : {len(clean_df)} / {len(df_raw)}")
print(f"  Patients uniques  : {clean_df['patient'].nunique()}")
print(f"  Taux de rejet     : {100*len(rejected_df)/len(df_raw):.1f}%")
print(f"\nLancer l'entraînement :")
print(f"  python main.py --data data/clean_dataset.json")